# SA-Jev · 环境与数据准备
先运行 MODE=pilot，产物保存到 MyDrive/SA-Jev/pilot；改为 full 后另存 full 目录。CPU 可运行，不训练 OpenJev-4B。pilot 每个源文件 64 条，不是完整论文数据，也不下载 2Wiki。流程成功生成本次 00_complete.json 后，默认刷新 Drive 写入并释放 Colab 运行时。首个代码单元的 AUTO_DISCONNECT=False 可关闭自动断开。

In [2]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Use 'full' after the pilot; separate Drive directories.
AUTO_DISCONNECT = True  # Release Colab after successful preparation and Drive flush.
from uuid import uuid4
PREPARATION_RUN_ID = uuid4().hex
ROOT = DRIVE / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
print('Code:', PROJECT, 'Artifacts:', ROOT)


Mounted at /content/drive
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/full


In [3]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)

import importlib, site

site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())

for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)

importlib.invalidate_caches()

import sa_jev
print('SA-Jev source:', sa_jev.__file__)

# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config()
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py
SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [4]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [5]:
import torch
from pathlib import Path
from tempfile import mkdtemp
from sa_jev.smoke import run

smoke_parent = ROOT / "smoke"
smoke_parent.mkdir(parents=True, exist_ok=True)

smoke_dir = Path(mkdtemp(prefix="run_", dir=smoke_parent))
print("本次测试目录：", smoke_dir)

summary = run(
    smoke_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
summary

本次测试目录： /content/drive/MyDrive/SA-Jev/full/smoke/run_4x7_2kn5
{'step': 1, 'epoch': 0, 'rank': 1.7917581796646118, 'suf': 0.6931471526622772, 'rel': 0.6931471526622772, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 0.9300143379998644, 'eta_seconds': 0.9300143379998644, 'grad_norm': 0.0001167683003586717, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 10, 'task': 2.6235347628593444, 'total_loss': 2.6235347628593444}
{'step': 2, 'epoch': 0, 'rank': 1.6094295978546143, 'suf': 0.693147212266922, 'rel': 0.6931336522102356, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 1.8401369409998551, 'eta_seconds': 0.0, 'grad_norm': 0.0002577165432740003, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 9, 'task': 2.4412035405635835, 'total_loss': 2.4412035405635835, 'validation': {'rank': 1.7005923390388489, 'suf': 0.69314674536

{'test_only': True,
 'output_directory': '/content/drive/MyDrive/SA-Jev/full/smoke/run_4x7_2kn5',
 'model': 'tiny causal encoder + actual upstream OpenJev DecisionHead/OptionSetInteractor',
 'data': 'synthetic fixtures',
 'device': 'cpu',
 'train_states': 6,
 'validation_states': 3,
 'preference_pairs': 6,
 'stage1_optimizer_steps': 2,
 'stage2_optimizer_steps': 2,
 'checkpoint_roundtrip': True,
 'evaluation_rows': 1,
 'evaluation_errors': 0,
 'peak_reserved_gib': 0.0,
 'not_verified': ['trained OpenJev-4B weights',
  'A100 80GB peak',
  'live LightRAG extraction',
  'official Jev API',
  'scientific performance']}

本地向量编码器：首次运行下载固定版本的 BAAI/bge-small-en-v1.5（384 维），缓存到 Drive/huggingface；之后复用。加载前校验官方权重大小和 SHA-256；损坏时隔离该缓存文件，并自动重下同一版本一次，校验仍失败则停止。不会删除原始数据或图谱。此检查不调用聊天/向量 API，自动使用 GPU，无 GPU 时用 CPU。文本抽取与最终回答继续使用 LLM API。

In [6]:
from sa_jev.embeddings import check_embeddings
embedding_check = await check_embeddings(config['retrieval'], ROOT/'embedding_check.json')
embedding_check

Loading local embedding: BAAI/bge-small-en-v1.5@5c38ec7c405ec4b44b94cc5a9bb96e735b38267a, device=cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Local embedding ready in 8.6s: dim=384, batch=16, max_length=512, normalized=True
Embedding check: (2, 384), finite=True, 0.14s, GPU peak reserved=0.000 GiB


{'backend': 'local',
 'model': 'BAAI/bge-small-en-v1.5',
 'revision': '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a',
 'dimension': 384,
 'device': 'cpu',
 'load_seconds': 20.241845369999965,
 'encode_seconds': 0.13932737899995118,
 'texts_per_second': 21.532020637530625,
 'document_norms': [1.0, 1.0],
 'toy_top1_correct': True,
 'peak_gpu_reserved_gib': 0,
 'finite_vectors': True,
 'test_only': True}

下载原始数据：pilot 每个文件 64 条；full 读取完整源。2Wiki 的官方 dev.json 另置于 ROOT/raw/2wiki_dev.json。MuSiQue 两种官方格式均受支持。

In [7]:
from time import perf_counter
from sa_jev.preparation import download_sources, download_2wiki, normalize_sources
prepare_started = perf_counter()
print(f'[1/3] Download HotpotQA and MuSiQue: MODE={MODE}, directory={ROOT / "raw"}', flush=True)
download_sources(ROOT/'raw', limit=64 if MODE=='pilot' else None)
if MODE=='full':
    print('[2/3] Download and extract official 2Wiki data...', flush=True)
    download_2wiki(ROOT/'raw')
else:
    print('[2/3] Pilot: skip 2Wiki.', flush=True)
print('[3/3] Normalize questions, split by question family, and deduplicate evidence...', flush=True)
questions, manifest = normalize_sources(ROOT/'raw', ROOT/'data')
print(f'Download/normalization complete in {perf_counter()-prepare_started:.1f}s. Graph indexing is the next cell.', flush=True)
manifest

{'sources': [{'path': 'hotpot_train.jsonl',
   'records': 90447,
   'sha256_content': 'd28fd309e812a86e2a0b5abb1eea34382c4cc7dcb860c3be2ec540ccbe8c84cd'},
  {'path': 'hotpot_validation.jsonl',
   'records': 7405,
   'sha256_content': 'a178c67fccd3c3d001a790a9a6c21722a6767a6812221d43601ebade58ad98bb'},
  {'path': 'musique_ans_train.jsonl',
   'records': 19938,
   'sha256_content': '2173d24a3bbf0b454293ba4932f6f144dec285c1ac90dbbaeb102235fb1ff36b'},
  {'path': 'musique_ans_dev.jsonl',
   'records': 2417,
   'sha256_content': '092034bdabc7a36262b5df85e651799782113b10adfdc400eea3968a878a1abf'},
  {'path': '2wiki_dev.json',
   'records': 12576,
   'sha256_content': 'ff858dc04e6be09cf3f3792a97d4087fd87dca0a9e8d1c5c50a1a5376fe33b1c'}],
 'question_fingerprint': '52cfacdfd6b4f5c0b60d4724b39e4d3d4127919c3e8624f019da2cf60beace0e',
 'corpus_fingerprint': '82ce3a8d7cda372656832826e61766ae3af16a089daf787d996d2c957da7e273',
 'questions': 132783,
 'corpus_documents': 657246,
 'splits': {'train': 93899

固定 LightRAG 混合召回。原始段落统一入库，不索引答案或标签。向量使用本地 BGE 编码器，文本抽取调用 LLM API。先检查 configs/research.yaml。旧 API 向量索引不能复用；切换建图模型或编码器后使用新的实验 ROOT，重建索引和候选池。首次建图通常比训练 pilot 更耗时。

In [8]:
from sa_jev.experiments import cache_pools
await cache_pools(ROOT, config)

Loading local embedding: BAAI/bge-small-en-v1.5@5c38ec7c405ec4b44b94cc5a9bb96e735b38267a, device=cpu


SafetensorError: Error while deserializing header: header too large

In [ ]:
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, dense=config['data']['dense_mining'])

训练前完整扫描公开 OpenJevData。检出的重叠保留记录并在论文报告；未检出不代表证明完全未见。此步骤不使用独立扰动测试集调参。

In [ ]:
cli('overlap')
cli('review-packets')

In [ ]:
from sa_jev.preparation import finish_preparation
completion = finish_preparation(ROOT, config, mode=MODE, run_id=PREPARATION_RUN_ID)
completion

核对实际 K 分布和有效标签数量。缺候选不复制补齐，未知相关性不硬判负例。若排序有效状态太少，核查候选注释映射/补充核验，不能把掩码简单删掉。把独立扰动草稿人工核验后保存为 perturbations_reviewed.jsonl。

结束后自动断开：只有本次运行已成功完成才执行。先刷新并卸载 Drive，再释放 Colab 运行时。Drive 刷新失败或流程失败时不主动释放；重新连接需重装环境并挂载 Drive，已保存数据保留。本地运行会跳过此步骤。

In [ ]:
from sa_jev.environment import disconnect_after_preparation
disconnect_after_preparation(ROOT, run_id=PREPARATION_RUN_ID, enabled=AUTO_DISCONNECT)